# Fixed Effects Analysis — Twitter Sentiment & Stock Returns
**Ashley Thompson — Capstone**

Models 1–4 + robustness checks + publication-ready LaTeX table export.

> No GPU needed — standard Colab CPU runtime is fine for this notebook.

## 1. Install dependencies

In [1]:
!pip install -q pyfixest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.2/607.2 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.3 MB/s eta 0:00:00


## 2. Load data
**Option A — Google Drive** | **Option B — Manual upload**  
Run one, skip the other.

In [10]:
# Option A — Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Update this path to wherever panel_long.csv lives in your Drive
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/Capstone/panel_long.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# GitHub output setup — run after whichever data option above
from google.colab import userdata
import os, subprocess

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_DIR     = '/content/Capstone'
OUTPUT_PATH  = REPO_DIR + '/output/'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone',
         f'https://{GITHUB_TOKEN}@github.com/ATHOMP-03/Capstone.git',
         REPO_DIR],
        check=True,
    )
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f'Output path ready: {OUTPUT_PATH}')

Output path ready: /content/Capstone/output/


## 3. Setup

In [12]:
import numpy as np
import pandas as pd
import pyfixest as pf
import os

np.random.seed(42)
os.makedirs(OUTPUT_PATH, exist_ok=True)

long = pd.read_csv(DATA_PATH, parse_dates=['date'])
long = long.sort_values(['ticker', 'date']).reset_index(drop=True)
print(f'Loaded {len(long):,} rows x {long.shape[1]} cols')
long.head()

Loaded 160,487 rows x 22 cols


,date,ticker,px_open,px_close,px_high,px_low,mkt_cap,total_equity,debt_to_equity,volume,...,news_sent,rsi_30,ma_50,twitter_neg_count,return,lag1,lag2,lag3,lag5,lag7
0,2025-01-01,A UN Equity,134.81,134.34,135.70,134.06,38366.8729,5898.0,60.5968,288566.0,...,-0.2921,45.9116,135.2216,0.0,-0.47,NaN,NaN,NaN,NaN,NaN
1,2025-01-02,A UN Equity,135.21,133.43,135.36,132.87,38106.9811,5898.0,60.5968,315308.0,...,-0.2921,44.9912,135.1550,0.0,-1.78,-0.47,NaN,NaN,NaN,NaN
2,2025-01-03,A UN Equity,133.45,135.69,136.04,132.86,38752.4265,5898.0,60.5968,399054.0,...,-0.3471,47.6856,135.1996,0.0,2.24,-1.78,-0.47,NaN,NaN,NaN
3,2025-01-06,A UN Equity,135.60,136.43,138.24,135.60,38963.7671,5898.0,60.5968,353095.0,...,-0.3471,48.5393,135.2676,0.0,0.83,2.24,-1.78,-0.47,NaN,NaN
4,2025-01-07,A UN Equity,136.83,137.41,140.28,136.83,39243.6504,5898.0,60.5968,359113.0,...,-0.2385,49.6647,135.4020,0.0,0.58,0.83,2.24,-1.78,NaN,NaN


## 4. Model 1 — Simple FE
Baseline within-firm estimate of Twitter sentiment on same-day return.

In [13]:
m1 = pf.feols('return ~ twitter_sent | ticker', data=long, vcov='hetero')
m1.summary()

###

Estimation:  OLS
Dep. var.: return, Fixed effects: ticker
sample: None = all
Inference:  hetero
Observations:  160103

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| twitter_sent  |     -0.370 |        0.230 |    -1.606 |      0.108 | -0.821 |   0.082 |
---
RMSE: 8.811 R2: 0.002 R2 Within: 0.0 


## 5. Model 2 — FE with confounder matrix
Adds fundamentals and technical indicators. Excludes `px_open` and `px_close` (collinear with return).

In [14]:
confounders = [
    'px_high', 'px_low', 'mkt_cap', 'total_equity', 'debt_to_equity',
    'volume', 'news_sent', 'rsi_30', 'ma_50', 'twitter_neg_count',
]
m2 = pf.feols(
    f"return ~ twitter_sent + {' + '.join(confounders)} | ticker",
    data=long, vcov='hetero'
)
m2.summary()

###

Estimation:  OLS
Dep. var.: return, Fixed effects: ticker
sample: None = all
Inference:  hetero
Observations:  147128

| Coefficient       |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| twitter_sent      |     -0.422 |        0.325 |    -1.298 |      0.194 | -1.059 |   0.215 |
| px_high           |     -0.085 |        0.121 |    -0.699 |      0.485 | -0.322 |   0.153 |
| px_low            |      0.091 |        0.122 |     0.743 |      0.457 | -0.149 |   0.331 |
| mkt_cap           |      0.000 |        0.000 |     0.865 |      0.387 | -0.000 |   0.000 |
| total_equity      |      0.000 |        0.000 |     0.247 |      0.805 | -0.000 |   0.000 |
| debt_to_equity    |     -0.000 |        0.000 |    -1.819 |      0.069 | -0.000 |   0.000 |
| volume            |      0.000 |        0.000 |     1.139 |      0.255 | -0.000 |   0.000 |
| news_sent         |     -1.0

## 6. Model 3 — Continuous negative sentiment treatment
`neg_twitter_sent = clip(twitter_sent, max=0)`: zero on positive days, raw score on negative days.

In [15]:
long['neg_twitter_sent'] = long['twitter_sent'].clip(upper=0)
m3 = pf.feols('return ~ neg_twitter_sent | ticker', data=long, vcov='hetero')
m3.summary()

###

Estimation:  OLS
Dep. var.: return, Fixed effects: ticker
sample: None = all
Inference:  hetero
Observations:  160103

| Coefficient      |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:-----------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| neg_twitter_sent |     -0.072 |        0.271 |    -0.266 |      0.790 | -0.603 |   0.459 |
---
RMSE: 8.811 R2: 0.002 R2 Within: 0.0 


## 7. Model 4 — Negative tweet count as standalone treatment
Per Teti et al. (2019): polarity-broken counts outperform total count. `twitter_neg_count` sidesteps neutral-tweet dilution in the Bloomberg index.

In [16]:
m4 = pf.feols('return ~ twitter_neg_count | ticker', data=long, vcov='hetero')
m4.summary()

###

Estimation:  OLS
Dep. var.: return, Fixed effects: ticker
sample: None = all
Inference:  hetero
Observations:  159232

| Coefficient       |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| twitter_neg_count |      0.001 |        0.001 |     0.549 |      0.583 | -0.001 |   0.002 |
---
RMSE: 8.834 R2: 0.002 R2 Within: 0.0 


## 8. Placebo test
Does today's `twitter_sent` predict *yesterday's* return (`lag1`)? Coefficient should be insignificant under a causal story.

In [17]:
placebo = pf.feols('lag1 ~ twitter_sent | ticker', data=long, vcov='hetero')
placebo.summary()

###

Estimation:  OLS
Dep. var.: lag1, Fixed effects: ticker
sample: None = all
Inference:  hetero
Observations:  159606

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| twitter_sent  |      2.348 |        0.225 |    10.450 |      0.000 |  1.908 |   2.789 |
---
RMSE: 8.816 R2: 0.003 R2 Within: 0.001 


## 9. No-reversal test — Gu & Kurov (2020)
All lagged sentiment variables estimated jointly. Only `sent_lag1` should be significant under a causal information story.

In [18]:
for n in [1, 2, 3, 5, 7]:
    long[f'sent_lag{n}'] = long.groupby('ticker')['twitter_sent'].shift(n)

no_reversal = pf.feols(
    'return ~ sent_lag1 + sent_lag2 + sent_lag3 + sent_lag5 + sent_lag7 | ticker',
    data=long, vcov='hetero'
)
no_reversal.summary()

###

Estimation:  OLS
Dep. var.: return, Fixed effects: ticker
sample: None = all
Inference:  hetero
Observations:  156589

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| sent_lag1     |     -0.322 |        0.196 |    -1.641 |      0.101 | -0.707 |   0.063 |
| sent_lag2     |      0.088 |        0.196 |     0.452 |      0.651 | -0.295 |   0.472 |
| sent_lag3     |     -0.225 |        0.192 |    -1.171 |      0.241 | -0.603 |   0.152 |
| sent_lag5     |     -0.285 |        0.175 |    -1.626 |      0.104 | -0.628 |   0.058 |
| sent_lag7     |      0.222 |        0.166 |     1.336 |      0.181 | -0.104 |   0.547 |
---
RMSE: 8.841 R2: 0.002 R2 Within: 0.0 


## 10. Export — Publication-ready LaTeX tables
Saves two `.tex` files to your output folder:
- `fe_main_results.tex` — Models 1–4 side by side
- `fe_robustness.tex` — Placebo and no-reversal test

In [21]:
# Main results table
main_tex = pf.etable(
    [m1, m2, m3, m4],
    type       = 'tex',
    signif_code = [0.01, 0.05, 0.1],
    signif_symbols = ['***', '**', '*'],
    notes       = 'Heteroskedasticity-robust SE in parentheses. Company fixed effects in all models. Model 3 treatment is clip(twitter\_sent, max=0). Model 4 per Teti et al. (2019).'
)
with open(OUTPUT_PATH + 'fe_main_results.tex', 'w') as f:
    f.write(main_tex)
print('Saved fe_main_results.tex')
print(main_tex)

Saved fe_main_results.tex
\begin{threeparttable}
\begingroup
\renewcommand\cellalign{t}
\renewcommand\arraystretch{1}
\setlength{\tabcolsep}{3pt}
\begin{tabularx}{\linewidth}{@{}>{\raggedright\arraybackslash}l>{\centering\arraybackslash}X>{\centering\arraybackslash}X>{\centering\arraybackslash}X>{\centering\arraybackslash}X}
\toprule
 & \multicolumn{4}{c}{return} \\
\cmidrule(lr){2-5}
 & (1) & (2) & (3) & (4) \\
\midrule
\addlinespace[1ex]
twitter_sent & \makecell{-0.37 \\ (0.23)} & \makecell{-0.422 \\ (0.325)} &  &  \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
px_high &  & \makecell{-0.085 \\ (0.121)} &  &  \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
px_low &  & \makecell{0.091 \\ (0.122)} &  &  \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
mkt_cap &  & \makecell{0 \\ (0.000001)} &  &  \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
total_equity &  & \makecell{0.000001 \\ (0.000004)} &  &  \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
debt_to_equity &  & \makecell{-0.000089 \\ (0.000049)}

<>:7: SyntaxWarning: invalid escape sequence '\_'
<>:7: SyntaxWarning: invalid escape sequence '\_'
/tmp/ipykernel_3291/2577676743.py:7: SyntaxWarning: invalid escape sequence '\_'
  notes       = 'Heteroskedasticity-robust SE in parentheses. Company fixed effects in all models. Model 3 treatment is clip(twitter\_sent, max=0). Model 4 per Teti et al. (2019).'


In [22]:
# Robustness table
robustness_tex = pf.etable(
    [placebo, no_reversal],
    type        = 'tex',
    signif_code = [0.01, 0.05, 0.1],
    signif_symbols = ['***', '**', '*'],
    notes       = 'Placebo: lag1 $\sim$ twitter\_sent $|$ ticker. No-reversal: all sentiment lags estimated jointly. Only sent\_lag1 should be significant under a causal interpretation (Gu \& Kurov, 2020).'
)
with open(OUTPUT_PATH + 'fe_robustness.tex', 'w') as f:
    f.write(robustness_tex)
print('Saved fe_robustness.tex')
print(robustness_tex)

Saved fe_robustness.tex
\begin{threeparttable}
\begingroup
\renewcommand\cellalign{t}
\renewcommand\arraystretch{1}
\setlength{\tabcolsep}{3pt}
\begin{tabularx}{\linewidth}{@{}>{\raggedright\arraybackslash}l>{\centering\arraybackslash}X>{\centering\arraybackslash}X}
\toprule
 & \multicolumn{1}{c}{lag1} & \multicolumn{1}{c}{return} \\
\cmidrule(lr){2-2} \cmidrule(lr){3-3}
 & (1) & (2) \\
\midrule
\addlinespace[1ex]
twitter_sent & \makecell{2.348 \\ (0.225)} &  \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
sent_lag1 &  & \makecell{-0.322 \\ (0.196)} \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
sent_lag2 &  & \makecell{0.088 \\ (0.196)} \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
sent_lag3 &  & \makecell{-0.225 \\ (0.192)} \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
sent_lag5 &  & \makecell{-0.285 \\ (0.175)} \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
sent_lag7 &  & \makecell{0.222 \\ (0.166)} \\
\addlinespace[0.5ex]
\midrule
\addlinespace[1ex]
ticker & x & x \\
\addlinespace[0.5ex]
\m

<>:7: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_3291/3097419905.py:7: SyntaxWarning: invalid escape sequence '\s'
  notes       = 'Placebo: lag1 $\sim$ twitter\_sent $|$ ticker. No-reversal: all sentiment lags estimated jointly. Only sent\_lag1 should be significant under a causal interpretation (Gu \& Kurov, 2020).'


In [28]:
print(f'Content of {OUTPUT_PATH}fe_main_results.tex:')
!cat {OUTPUT_PATH}fe_main_results.tex

Content of /content/Capstone/output/fe_main_results.tex:
\begin{threeparttable}
\begingroup
\renewcommand\cellalign{t}
\renewcommand\arraystretch{1}
\setlength{\tabcolsep}{3pt}
\begin{tabularx}{\linewidth}{@{}>{\raggedright\arraybackslash}l>{\centering\arraybackslash}X>{\centering\arraybackslash}X>{\centering\arraybackslash}X>{\centering\arraybackslash}X}
\toprule
 & \multicolumn{4}{c}{return} \\
\cmidrule(lr){2-5}
 & (1) & (2) & (3) & (4) \\
\midrule
\addlinespace[1ex]
twitter_sent & \makecell{-0.37 \\ (0.23)} & \makecell{-0.422 \\ (0.325)} &  &  \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
px_high &  & \makecell{-0.085 \\ (0.121)} &  &  \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
px_low &  & \makecell{0.091 \\ (0.122)} &  &  \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
mkt_cap &  & \makecell{0 \\ (0.000001)} &  &  \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
total_equity &  & \makecell{0.000001 \\ (0.000004)} &  &  \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
debt_to_equity &  & \ma

In [29]:
print(f'\nContent of {OUTPUT_PATH}fe_robustness.tex:')
!cat {OUTPUT_PATH}fe_robustness.tex


Content of /content/Capstone/output/fe_robustness.tex:
\begin{threeparttable}
\begingroup
\renewcommand\cellalign{t}
\renewcommand\arraystretch{1}
\setlength{\tabcolsep}{3pt}
\begin{tabularx}{\linewidth}{@{}>{\raggedright\arraybackslash}l>{\centering\arraybackslash}X>{\centering\arraybackslash}X}
\toprule
 & \multicolumn{1}{c}{lag1} & \multicolumn{1}{c}{return} \\
\cmidrule(lr){2-2} \cmidrule(lr){3-3}
 & (1) & (2) \\
\midrule
\addlinespace[1ex]
twitter_sent & \makecell{2.348 \\ (0.225)} &  \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
sent_lag1 &  & \makecell{-0.322 \\ (0.196)} \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
sent_lag2 &  & \makecell{0.088 \\ (0.196)} \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
sent_lag3 &  & \makecell{-0.225 \\ (0.192)} \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
sent_lag5 &  & \makecell{-0.285 \\ (0.175)} \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
sent_lag7 &  & \makecell{0.222 \\ (0.166)} \\
\addlinespace[0.5ex]
\midrule
\addlinespace[1ex]
ticker & 

In [30]:
# Push outputs to GitHub
import os, subprocess

os.chdir(REPO_DIR)
subprocess.run(['git', 'config', 'user.email', 'ATHOMP-03@users.noreply.github.com'])
subprocess.run(['git', 'config', 'user.name',  'ATHOMP-03'])
subprocess.run(['git', 'add', 'output/'])
result = subprocess.run(['git', 'diff', '--cached', '--quiet'])
if result.returncode != 0:
    subprocess.run(['git', 'commit', '-m', 'Add FE analysis outputs from Colab'], check=True)
    subprocess.run(['git', 'push'], check=True)
    print('Pushed to GitHub.')
else:
    print('Nothing new to commit.')

Nothing new to commit.


In [32]:
import subprocess, os
os.chdir(REPO_DIR)
result = subprocess.run(['git', 'push'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)


To https://github.com/ATHOMP-03/Capstone.git
   2c3cabd..44074b0  main -> main

